In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
import re
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from gensim.models import KeyedVectors  # Gensim for Word2Vec
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


In [10]:
pip install Sastrawi

In [11]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [12]:
# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/TA DIAZ/datasetdiaz.csv')

# Drop missing values
df = df.dropna(subset=['Tweet', 'HS'])
X = np.array(df['Tweet'])
y = np.array(df['HS'])

In [14]:
# Split data for Word2Vec training (90%) and SVM training (10%)
X_w2v_train, X_svm, y_w2v_train, y_svm = train_test_split(X, y, test_size=0.1, random_state=42)

In [15]:
# Tokenization function
def tokenize_text(text):
    return word_tokenize(text)

# Tokenize Word2Vec training data
X_train_tokenized = [tokenize_text(text) for text in X_w2v_train]

In [20]:
# Load pre-trained Word2Vec model
model_word2vec = KeyedVectors.load_word2vec_format('/content/drive/MyDrive/TA DIAZ/ws_model100.bin', binary=True)

In [21]:
# Function to get Word2Vec features
def get_w2v_features(w2v_model, sentence_group):
    feature_vec = np.zeros(w2v_model.vector_size, dtype="float32")
    num_words = 0
    for word in sentence_group:
        if word in w2v_model:
            feature_vec = np.add(feature_vec, w2v_model[word])
            num_words += 1
    if num_words > 0:
        feature_vec = np.divide(feature_vec, num_words)
    return feature_vec

In [22]:
# Convert Word2Vec features
X_train_w2v = [get_w2v_features(model_word2vec, tokens) for tokens in X_train_tokenized]

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_w2v)

# Split SVM dataset
X_svm_train, X_svm_test, y_svm_train, y_svm_test = train_test_split(X_svm, y_svm, test_size=0.1, random_state=42)

# Tokenize and transform testing data
X_test_tokenized = [tokenize_text(text) for text in X_svm_test]
X_test_w2v = [get_w2v_features(model_word2vec, tokens) for tokens in X_test_tokenized]
X_test_scaled = scaler.transform(X_test_w2v)

In [23]:
# Train and evaluate multiple SVM kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results = {}
for kernel in kernels:
    print(f"\nTraining SVM with {kernel} kernel...")
    classifier = SVC(kernel=kernel)
    classifier.fit(X_train_scaled, y_w2v_train)
    y_pred = classifier.predict(X_test_scaled)

    # Compute evaluation metrics
    accuracy = accuracy_score(y_svm_test, y_pred)
    precision = precision_score(y_svm_test, y_pred)
    recall = recall_score(y_svm_test, y_pred)
    f1 = f1_score(y_svm_test, y_pred)
    cm = confusion_matrix(y_svm_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Store results
    results[kernel] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm
    }

    # Print results
    print(f"Confusion Matrix ({kernel} Kernel):\n{cm}")
    print(f"\nMetrics:")
    print(f"Akurasi      : {accuracy:.4f}")
    print(f"Precision    : {precision:.4f} (Dari yang diprediksi sebagai Hate Speech, berapa yang benar-benar Hate Speech)")
    print(f"Recall       : {recall:.4f} (Dari seluruh Hate Speech, berapa yang berhasil ditemukan oleh model)")
    print(f"F1-Score     : {f1:.4f}")
    print(f"\nConfusion Matrix Breakdown:")
    print(f"True Negative (TN) : {tn} (Non-Hate Speech diklasifikasikan dengan benar)")
    print(f"False Positive (FP): {fp} (Non-Hate Speech diklasifikasikan sebagai Hate Speech)")
    print(f"False Negative (FN): {fn} (Hate Speech diklasifikasikan sebagai Non-Hate Speech)")
    print(f"True Positive (TP) : {tp} (Hate Speech diklasifikasikan dengan benar)")



Training SVM with linear kernel...
Confusion Matrix (linear Kernel):
[[53 13]
 [17 32]]

Metrics:
Akurasi      : 0.7391
Precision    : 0.7111 (Dari yang diprediksi sebagai Hate Speech, berapa yang benar-benar Hate Speech)
Recall       : 0.6531 (Dari seluruh Hate Speech, berapa yang berhasil ditemukan oleh model)
F1-Score     : 0.6809

Confusion Matrix Breakdown:
True Negative (TN) : 53 (Non-Hate Speech diklasifikasikan dengan benar)
False Positive (FP): 13 (Non-Hate Speech diklasifikasikan sebagai Hate Speech)
False Negative (FN): 17 (Hate Speech diklasifikasikan sebagai Non-Hate Speech)
True Positive (TP) : 32 (Hate Speech diklasifikasikan dengan benar)

Training SVM with poly kernel...
Confusion Matrix (poly Kernel):
[[53 13]
 [22 27]]

Metrics:
Akurasi      : 0.6957
Precision    : 0.6750 (Dari yang diprediksi sebagai Hate Speech, berapa yang benar-benar Hate Speech)
Recall       : 0.5510 (Dari seluruh Hate Speech, berapa yang berhasil ditemukan oleh model)
F1-Score     : 0.6067

Co